In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os
from collections import defaultdict
import pickle

In [ ]:
import pyterrier as pt
pt.init()

# Load data

In [ ]:
qrel_path = "../data/CAST_qrels/qrels-docs.2019.txt"
qrels_df = pd.read_csv(qrel_path, delimiter=" ", header=None)
qrels_df[[3]] = qrels_df[[3]].astype(int)
qrels_df = qrels_df.drop([1], axis=1)
qrels_df.columns=["qid", "docno", "label"]
qrels = qrels_df

In [ ]:
topics_path='../data/CAST-2019/test_manual_utterance.tsv' #manual

topics_df = pd.read_csv(topics_path, delimiter="\t", header=None)
topics_df.columns=["qid", "query"]
topics = topics_df
topics.head()

# Retrieval effectiveness by cache size

Evaluates STAR (L2) first-utterance cache strategy across cache sizes.

In [ ]:
topk = 1000
results_path = ("../data/star-ranking/"
                "CAST-manual-queries-star-L2-ranking-top1000-cache-top"
                + str(topk) + "-first-utt_new.tsv")
results_df = pd.read_csv(results_path, delimiter="\t", header=None)
results_df[3] = 1000 - results_df[2]
results_df.columns = ["qid", "docno", "rank", "score"]
results_df = results_df.loc[results_df["rank"] < 201]
results_df.head()

In [ ]:
%%time
pt.Experiment([results_df], topics, qrels, names=["STAR"], 
              eval_metrics=["map", "recip_rank", "recall_200", "P_3", "P_1", "ndcg_cut_3"])

In [ ]:
topk = 2000
results_path = ("../data/star-ranking/"
                "CAST-manual-queries-star-L2-ranking-top1000-cache-top"
                + str(topk) + "-first-utt_new.tsv")
results_df = pd.read_csv(results_path, delimiter="\t", header=None)
results_df[3] = 1000 - results_df[2]
results_df.columns = ["qid", "docno", "rank", "score"]
results_df = results_df.loc[results_df["rank"] < 201]
results_df.head()

In [ ]:
%%time
pt.Experiment([results_df], topics, qrels, names=["STAR"], 
              eval_metrics=["map", "recip_rank", "recall_200", "P_3", "P_1", "ndcg_cut_3"])

In [ ]:
topk = 5000
results_path = ("../data/star-ranking/"
                "CAST-manual-queries-star-L2-ranking-top1000-cache-top"
                + str(topk) + "-first-utt_new.tsv")
results_df = pd.read_csv(results_path, delimiter="\t", header=None)
results_df[3] = 1000 - results_df[2]
results_df.columns = ["qid", "docno", "rank", "score"]
results_df = results_df.loc[results_df["rank"] < 201]
results_df.head()

In [ ]:
%%time
pt.Experiment([results_df], topics, qrels, names=["STAR"], 
              eval_metrics=["map", "recip_rank", "recall_200", "P_3", "P_1", "ndcg_cut_3"])

In [ ]:
topk = 10000
results_path = ("../data/star-ranking/"
                "CAST-manual-queries-star-L2-ranking-top1000-cache-top"
                + str(topk) + "-first-utt_new.tsv")
results_df = pd.read_csv(results_path, delimiter="\t", header=None)
results_df[3] = 1000 - results_df[2]
results_df.columns = ["qid", "docno", "rank", "score"]
results_df = results_df.loc[results_df["rank"] < 201]
results_df.head()

In [ ]:
%%time
pt.Experiment([results_df], topics, qrels, names=["STAR"], 
              eval_metrics=["map", "recip_rank", "recall_200", "P_3", "P_1", "ndcg_cut_3"])

# Coverage analysis

Approximated coverage: how many of the top-k ground-truth neighbours for query q_b lie within the cache built for q_a.

In [ ]:
topk_vals = [1000, 2000, 5000, 10000]

print("=== First-utterance cache (STAR L2) ===\n")
for k in topk_vals:
    path = f"../data/star-ranking/approximated-coverage-star-L2-ranking-top1000-cache-top{k}_first_utt.tsv"
    rows = []
    with open(path) as fh:
        for line in fh:
            parts = line.strip().split("\t")
            covs = [int(x) for x in parts[1].strip("()").split(",")]
            rows.append(covs)
    df = pd.DataFrame(rows, columns=["cov@3", "cov@5", "cov@10"])
    print(f"cache-top{k}:")
    print(df.mean().round(3).to_string(), "\n")

In [ ]:
print("=== Cache with update (STAR L2) ===\n")
for k in topk_vals:
    path = f"../data/star-ranking/approximated-coverage-star-L2-ranking-top1000-cache-top{k}_with_update.tsv"
    rows = []
    with open(path) as fh:
        for line in fh:
            parts = line.strip().split("\t")
            covs = [int(x) for x in parts[1].strip("[]").split(",")]
            rows.append(covs)
    df = pd.DataFrame(rows, columns=["cov@3", "cov@5", "cov@10"])
    print(f"cache-top{k}:")
    print(df.mean().round(3).to_string(), "\n")